In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Project 9 — Local Study Notes Generator
## Turn Raw Text into Cornell Notes, Flashcards, and Quizzes

**Stack:** Ollama · LangChain · Pydantic · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic

## Step 1 — Setup

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field

llm = ChatOllama(model="qwen3:8b", temperature=0.2)

study_material = """
Operating Systems — Process Management

A process is a program in execution. Each process has a Process Control Block (PCB)
containing: process state, program counter, CPU registers, memory management info,
and I/O status.

Process States: New → Ready → Running → Waiting → Terminated.
The scheduler moves processes between Ready and Running queues.

Context switching saves the state of the current process and loads the state of
the next. It involves saving/loading PCBs and is pure overhead — no useful work
is done during a context switch. Typical cost: 1-10 microseconds.

Scheduling Algorithms:
- FCFS (First Come First Served): Simple but causes convoy effect
- SJF (Shortest Job First): Optimal average wait time but requires knowing burst time
- Round Robin: Fair, uses time quantum. Too small = overhead, too large = FCFS
- Priority Scheduling: Risk of starvation, solved with aging
"""
print(f"Study material loaded: {len(study_material)} chars")

## Step 2 — Generate Cornell Notes

In [ ]:
cornell_prompt = ChatPromptTemplate.from_messages([
    ("system", """Generate Cornell Notes format from the study material:

CUES (left column) | NOTES (right column)
Key questions/terms | Detailed notes, examples, diagrams
                   |
─────────────────────────────────────────
SUMMARY: 3-5 sentence summary of the entire topic

Rules:
- Cues should be questions that the notes answer
- Notes should be concise bullet points
- Include examples where applicable
- Summary should capture the main concepts"""),
    ("human", "Generate Cornell Notes for:\n\n{material}")
])

cornell_chain = cornell_prompt | llm | StrOutputParser()
cornell_notes = cornell_chain.invoke({"material": study_material})
print(cornell_notes)

## Step 3 — Generate Flashcards

In [ ]:
class Flashcard(BaseModel):
    front: str = Field(description="Question or term")
    back: str = Field(description="Answer or definition")
    difficulty: str = Field(description="easy, medium, hard")

class FlashcardDeck(BaseModel):
    topic: str
    cards: list[Flashcard]

flashcard_llm = llm.with_structured_output(FlashcardDeck)
deck = flashcard_llm.invoke(
    f"Generate 10 flashcards from this material:\n\n{study_material}"
)
print(f"Topic: {deck.topic}")
print(f"Generated {len(deck.cards)} flashcards\n")
for i, card in enumerate(deck.cards):
    print(f"Card {i+1} [{card.difficulty}]:")
    print(f"  Q: {card.front}")
    print(f"  A: {card.back}\n")

## Step 4 — Generate Practice Quiz

In [ ]:
class QuizQuestion(BaseModel):
    question: str
    options: list[str] = Field(description="4 options (A-D)")
    correct_answer: str = Field(description="Letter of correct option")
    explanation: str

class Quiz(BaseModel):
    questions: list[QuizQuestion]

quiz_llm = llm.with_structured_output(Quiz)
quiz = quiz_llm.invoke(
    f"Generate 5 multiple choice questions:\n\n{study_material}"
)
print("=== PRACTICE QUIZ ===\n")
for i, q in enumerate(quiz.questions):
    print(f"Q{i+1}: {q.question}")
    for opt in q.options:
        print(f"   {opt}")
    print(f"   Answer: {q.correct_answer}")
    print(f"   Explanation: {q.explanation}\n")

## Step 5 — Study Session Simulator

In [ ]:
def study_session(material, num_rounds=2):
    """Run an automated study → quiz → review cycle."""
    print("=== STUDY SESSION START ===\n")

    # Round 1: Key concepts
    concepts_prompt = ChatPromptTemplate.from_messages([
        ("system", "List the 5 most important concepts from this material, "
         "each with a one-sentence explanation."),
        ("human", "{material}")
    ])
    concepts = (concepts_prompt | llm | StrOutputParser()).invoke({"material": material})
    print("KEY CONCEPTS TO LEARN:")
    print(concepts)

    # Round 2: Self-test
    quiz = quiz_llm.invoke(f"Generate 3 quiz questions:\n\n{material}")
    print("\n=== SELF-TEST ===")
    score = 0
    for i, q in enumerate(quiz.questions):
        print(f"\nQ{i+1}: {q.question}")
        for opt in q.options:
            print(f"  {opt}")
        print(f"  Correct: {q.correct_answer} — {q.explanation}")

    print(f"\n=== SESSION COMPLETE ===")

study_session(study_material)

## What You Learned
- **Cornell Notes** generation from raw text
- **Flashcard decks** with structured output
- **Quiz generation** with explanations
- **Study session automation** combining multiple techniques